# Short Tandem Repeat Expansions


**Version**: 1.0.0   
**Date**: 31-DEC-2025


**Description:**  
This notebook identifies short tandem repeat expansions with ExpansionHunter and concatenates results to filter pathogenic ranges.

**Table of contents:**  

1. Load packages
2. Define paths for input and output files
3. Get list of .cram absolute paths
4. Modify variant catalog file to run ataxia and chorea genes
5. Run ExpansionHuner
6. Summarize results
7. Filter by ranges
    * Autosomal dominant inheritance
    * Autosomal recessive inheritance

## Versions

ExpansionHunter-v3.1.2

## Load packages

In [ ]:
import os
from pathlib import Path
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess
import json
import pandas as pd

In [ ]:
d = date.today()

print(f'''
pandas=={pd.__version__}
''')

## Define paths for input and out files

In [ ]:
# CRAM input directory
cram_directory = Path('/home/crams')

# Reference (hg38)
reference = '/home/fasta/hg38/Homo_sapiens_assembly38.fasta'

# Variant Catalog (ExpansionHunter)
variant_catalog = '/home/ExpansionHunter-v3.1.2-linux_x86_64/variant_catalog/hg38/variant_catalog.json'

# Output directory
output_dir = Path('/home/output_str')
output_dir.mkdir(exist_ok=True)

## Get list of .cram absolute paths

In [ ]:
# Get absolute paths for all .cram files
cramList = [str(f.resolve()) 
            for f in cram_directory.glob('*_Cram/**/*.cram') 
            if f.is_file()]

In [ ]:
print('Number of samples: ', len(cramList))

## Modify variant catalog file to run ataxia and chorea genes


In [ ]:
# Open original variant catalog
with open(variant_catalog) as f:
    data = json.load(f)

In [ ]:
# Define stems for genes of interest
genes_of_interest = {
    'ATN1',     # DRPLA,   AD
    'ATX',      # SCA1,2,3,7,8,10 AD
    'FXN',      # FRDA,    AR 
    'HTT',      # HD,      AD
    'JPH3',     # HD-like, AD
    'PPP2R2B',  # SCA12,   AD
    'TBP',      # SCA17,   AD
    'CACNA1A',  # SCA6,    AD
}

In [ ]:
# Get filtered json
filtered = [entry for entry in data if any(gene in entry.get('LocusId', '') for gene in genes_of_interest)]

In [ ]:
# Save
with open(f'{output_dir}/variant_catalog.json', 'w') as f:
    json.dump(filtered, f, indent=2)

## Run ExpansionHunter

In [ ]:
# Define function to run ExpansionHunter
def run_expansion(inputCram):
    inputCram = Path(inputCram)
    out_file = output_dir / inputCram.stem
    cmd = [
        'ExpansionHunter',
        '--reads', str(inputCram),
        '--reference', str(reference),
        '--variant-catalog', str(output_dir / 'variant_catalog.json'),
        '--output-prefix', str(out_file)
    ]
    # Run per each sample
    try:
        subprocess.run(cmd, check=True)
        return inputCram.stem, None
    # Print error and continue
    except Exception as e:
        return inputCram.stem, e

In [ ]:
# Run in parallel
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = [executor.submit(run_expansion, f) for f in cramList]
    for future in as_completed(futures):
        sample, error = future.result()
        if error:
            print(f'Error: {sample} -> {error}')
        else:
            print(f'Completed: {sample}')

## Summarize results

In [ ]:
# Get all .json output files
all_data = {}
files = [f for f in os.listdir(output_dir) if f.startswith('COHORT_ID') and f.endswith('.json')]

In [ ]:
os.chdir(output_dir)

In [ ]:
# Retrieve gene name, repeat units and genotypes per sample
for f in files:
    sample = os.path.splitext(f)[0]
    with open(f) as jf:
        j = json.load(jf)
        if isinstance(j, list):
            for item in j:
                locus_results = item.get('LocusResults', {})
                for locus_key, locus_val in locus_results.items():
                    variants = locus_val.get('Variants', {})
                    for var_key, var_val in variants.items():
                        gene = var_val.get('VariantId')
                        repeat = var_val.get('RepeatUnit')
                        gt = var_val.get('Genotype')
                        if gene and repeat and gt:
                            key = (gene, repeat)
                            if key not in all_data:
                                all_data[key] = {}
                            all_data[key][sample] = gt
        else:
            locus_results = j.get('LocusResults', {})
            for locus_key, locus_val in locus_results.items():
                variants = locus_val.get('Variants', {})
                for var_key, var_val in variants.items():
                    gene = var_val.get('VariantId')
                    repeat = var_val.get('RepeatUnit')
                    gt = var_val.get('Genotype')
                    if gene and repeat and gt:
                        key = (gene, repeat)
                        if key not in all_data:
                            all_data[key] = {}
                        all_data[key][sample] = gt

In [ ]:
# Create pandas df
df = pd.DataFrame.from_dict(all_data, orient='index')
df.index = pd.MultiIndex.from_tuples(df.index, names=['Gene', 'Repeat'])
df.reset_index(inplace=True)
df

In [ ]:
# Save as an excel sheet
df.to_excel(f'{output_dir}/expansionhunter_output.xlsx', engine='openpyxl', index=False)

## Filter by ranges

### Autosomal dominant inheritance

In [ ]:
ad_pathogenic_ranges = {
    'ATN1': 49,
    'ATXN1': 39,
    'ATXN2': 35,
    'ATXN3': 56,
    'ATXN7': 36,
    'ATXN8OS': 80,
    'ATXN10': 800,
    'HTT': 36,
    'JPH3': 41,
    'PPP2R2B': 51,
    'TBP': 43,
    'CACNA1A': 21,
} 

In [ ]:
dominant_output = {}

for _, row in df.iterrows():
    gene = row['Gene']
    threshold = ad_pathogenic_ranges.get(gene)
    if threshold is None:
        continue
    gene_dict = {}
    for sample in df.columns:
        if sample == 'Gene':
            continue
        val = str(row[sample])
        if '/' in val:
            nums = [int(x) for x in val.split('/')]
            if any(n >= threshold for n in nums):
                gene_dict[sample] = val
    if gene_dict:
        dominant_output[gene] = gene_dict

In [ ]:
# Save as json
with open(f"{output_dir}/dominant_pathogenic_samples.json", "w") as f:
    json.dump(dominant_output, f, indent=2)

### Autosomal recessive inheritance

In [ ]:
ar_pathogenic_ranges = {
    'FXN': 70,
} 

In [ ]:
recessive_output = {}

for _, row in df.iterrows():
    gene = row['Gene']
    threshold = ar_pathogenic_ranges.get(gene)
    if threshold is None:
        continue
    gene_dict = {}
    for sample in df.columns:
        if sample == 'Gene':
            continue
        val = str(row[sample])
        if '/' in val:
            nums = [int(x) for x in val.split('/')]
            if all(n >= threshold for n in nums):
                gene_dict[sample] = val
    if gene_dict:
        recessive_output[gene] = gene_dict

In [ ]:
# Save as json
with open(f"{output_dir}/recessive_pathogenic_samples.json", "w") as f:
    json.dump(recessive_output, f, indent=2)

### Visualization with graphalignmentviewer


After expansion hunter is completed

In [ ]:
from collections import defaultdict
# ! unset MPLBACKEND
sample_locus = defaultdict(list)

with open("input.txt") as f:
    for line in f:
        sample, locus = line.strip().split()
        sample_locus[sample].append(locus)
env_name = "your_conda_env"
        
for sampleID, loci in sample_locus.items():
    for locus in loci:
        ! unset MPLBACKEND ; conda init ; conda activate env_name;  python3 /home/GraphAlignmentViewer/GraphAlignmentViewer.py \
            --variant_catalog variant_catalog_stripy.json \
            --read_align {sampleID}_realigned.bam \
            --gt_file {sampleID}.vcf \
            --locus_id {locus} \
            --output_prefix {sampleID}_{locus} \
            --dpi 300